[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ZeruiW/frontier-ai-courses/blob/main/C48_Cloud_Deployment_Course/04_rollout_autoscaling/04_rollout_autoscaling.ipynb)

# 04 · 发布策略与自动扩缩（滚动容量曲线、序贯检验、HPA 震荡，全部从零仿真）

目标：把 **滚动更新的容量凹陷 → 金丝雀判据的统计学 → HPA 控制律与震荡 → 热备容量** 从零写出来，
每个机制都**对拍**朴素参考、每个策略都**算一笔账**。

路线：容量曲线仿真 → peeking 的假阳性灾难 → 序贯检验修复 → HPA 闭环仿真（震荡/阻尼）→ 热备公式 → ✏️ 练习 → 📖 答案 → 🧪 发布策略成本胶囊。

> 心智模型：**发布 = 压小「爆炸半径 × MTTR」；扩缩 = 一个带纯延迟的反馈控制系统**。
> 两者的最优解都被同一个数字支配：**冷启动时间**。

## 1 · 滚动更新：容量凹陷有多深、多久

`maxUnavailable` 决定凹陷**多深**，`maxSurge` 决定更新**多快**、要多付多少 GPU。
LLM 的新 Pod 要 3 分钟才就绪，所以这个凹陷不是理论问题。

In [ ]:
import math, random
from dataclasses import dataclass, field

def rolling_update(desired, max_unavailable, max_surge, ready_delay_s, tick_s=10, horizon_s=1800):
    '''仿真滚动更新，返回每个时刻的 (可用旧副本+可用新副本) 列表。'''
    old_ready, new_pending, new_ready = desired, [], 0
    avail_trace, total_trace, t = [], [], 0
    while t < horizon_s and (old_ready > 0 or new_ready < desired):
        # 新 Pod 到点就绪
        arrived = [p for p in new_pending if p <= t]
        new_pending = [p for p in new_pending if p > t]
        new_ready += len(arrived)
        avail = old_ready + new_ready
        total = old_ready + new_ready + len(new_pending)
        # 摘旧：在不违反 maxUnavailable 的前提下
        while old_ready > 0 and avail - 1 >= desired - max_unavailable and new_ready + old_ready > desired - max_unavailable:
            if new_ready + old_ready - 1 < desired - max_unavailable:
                break
            old_ready -= 1; avail -= 1; total -= 1
            if new_ready >= desired: break
        # 起新：在不违反 maxSurge 的前提下
        while total < desired + max_surge and new_ready + len(new_pending) < desired:
            new_pending.append(t + ready_delay_s); total += 1
        avail_trace.append(avail); total_trace.append(total)
        t += tick_s
    return avail_trace, total_trace, t

DESIRED, READY_DELAY = 13, 180
for mu_, ms_ in [(3, 3), (0, 3), (0, 1), (3, 0)]:
    av, tot, dur = rolling_update(DESIRED, mu_, ms_, READY_DELAY)
    print(f'maxUnavailable={mu_}, maxSurge={ms_}: '
          f'最深容量 {min(av):>3d}/{DESIRED}, 峰值占用 {max(tot):>3d}, 更新耗时 {dur/60:>5.1f} 分钟')

av0, _, dur0 = rolling_update(DESIRED, 0, 3, READY_DELAY)
av3, _, dur3 = rolling_update(DESIRED, 3, 3, READY_DELAY)
assert min(av0) >= DESIRED, 'maxUnavailable=0 时容量不应下探'
assert min(av3) < DESIRED,  'maxUnavailable=3 时容量会下探'
print('\n✅ maxUnavailable=0 是 LLM 服务的标准做法：容量永不下探，代价是必须有 maxSurge 的空闲卡')

### maxSurge 的定价：付多少张卡换多快的更新

In [ ]:
print(f"{'maxSurge':>9s} {'更新耗时(min)':>14s} {'额外GPU·小时':>14s} {'额外成本$':>11s}")
GPU_PER_REPLICA, GPU_HOURLY = 2, 4.0
rows = []
for ms_ in [1, 2, 3, 5, 8, 13]:
    av, tot, dur = rolling_update(DESIRED, 0, ms_, READY_DELAY)
    extra_gpu_h = ms_ * GPU_PER_REPLICA * (dur / 3600)
    rows.append((ms_, dur / 60, extra_gpu_h * GPU_HOURLY))
    print(f'{ms_:>9d} {dur/60:>14.1f} {extra_gpu_h:>14.2f} {extra_gpu_h*GPU_HOURLY:>11.2f}')

durs = [r[1] for r in rows]
assert durs == sorted(durs, reverse=True), 'maxSurge 越大更新越快'
assert rows[0][1] > 3 * rows[-1][1], 'maxSurge=1（串行）应比全量并行慢 3 倍以上'
print(f'\n✅ maxSurge 1→13: 更新从 {rows[0][1]:.0f} 分钟压到 {rows[-1][1]:.0f} 分钟，'
      f'额外成本仅 ${rows[-1][2]:.1f}。')
print('   更新速度在这里是**极其便宜**的 —— 前提是集群有空闲卡放得下 surge。')

## 2 · 金丝雀判据的统计学：peeking 会毁掉你的判断

金丝雀天然是「持续监控、随时可能叫停」= **反复做检验**。
固定样本量的 p 值在反复偷看下会严重膨胀假阳性率。先把灾难跑出来。

In [ ]:
def two_prop_z(x1, n1, x2, n2):
    '''两比例 z 检验，返回 |z|。'''
    if n1 == 0 or n2 == 0: return 0.0
    p1, p2 = x1/n1, x2/n2
    p = (x1 + x2) / (n1 + n2)
    se = math.sqrt(p * (1-p) * (1/n1 + 1/n2))
    return 0.0 if se == 0 else abs(p1 - p2) / se

def naive_peeking_trial(err_rate_base, err_rate_canary, n_total=4000, peek_every=100, z_crit=1.96, seed=0):
    '''模拟一次金丝雀：每 peek_every 个样本看一次，一旦 |z|>z_crit 就叫停。'''
    rng = random.Random(seed)
    x1 = x2 = n1 = n2 = 0
    for i in range(1, n_total + 1):
        n1 += 1; x1 += rng.random() < err_rate_base
        n2 += 1; x2 += rng.random() < err_rate_canary
        if i % peek_every == 0 and two_prop_z(x1, n1, x2, n2) > z_crit:
            return True          # 叫停（判定为「有差异」）
    return False

# A/A 测试：新旧版本**完全相同**，任何「发现差异」都是假阳性
RATE = 0.01
fp_peek = sum(naive_peeking_trial(RATE, RATE, seed=s) for s in range(500)) / 500
# 对照：只在最后看一次（固定样本量，正确的 5%）
fp_once = sum(naive_peeking_trial(RATE, RATE, peek_every=4000, seed=s) for s in range(500)) / 500
print(f'A/A 测试（两版本完全相同）的假阳性率:')
print(f'  每 100 样本偷看一次: {fp_peek:.1%}   ← 名义上应该是 5%')
print(f'  只在结束时看一次  : {fp_once:.1%}')
assert fp_peek > 2 * fp_once, 'peeking 应显著抬高假阳性率'
print(f'\n⚠️  偷看把假阳性率抬高了 {fp_peek/max(fp_once,1e-9):.1f} 倍 ——')
print('   意味着大量**完全正常的发布**会被误判为坏版本而自动回滚。')

### 修复一：序贯概率比检验（SPRT）

SPRT 在**任意停止时刻**都保持名义错误率。它维护一个对数似然比，
越过上界判 H1、越过下界判 H0，中间继续观察。

In [ ]:
def sprt_trial(p0, p1, err_base, err_canary, alpha=0.05, beta=0.20, n_max=4000, seed=0):
    '''SPRT: H0 = 金丝雀错误率 p0（没变差）, H1 = p1（变差了）。
       返回 ('reject_h0' | 'accept_h0' | 'inconclusive', 用了多少样本)。'''
    rng = random.Random(seed)
    A = math.log((1 - beta) / alpha)        # 上界：判 H1
    B = math.log(beta / (1 - alpha))        # 下界：判 H0
    llr = 0.0
    for i in range(1, n_max + 1):
        x = 1 if rng.random() < err_canary else 0
        llr += math.log(p1/p0) if x else math.log((1-p1)/(1-p0))
        if llr >= A: return 'reject_h0', i      # 确认变差 -> 回滚
        if llr <= B: return 'accept_h0', i      # 确认没变差 -> 放行
    return 'inconclusive', n_max

P0, P1 = 0.01, 0.03      # H0: 1% 错误率（正常）; H1: 3%（明显变差）
aa = [sprt_trial(P0, P1, P0, P0, seed=s) for s in range(500)]      # A/A：真的没变差
ab = [sprt_trial(P0, P1, P0, P1, seed=s) for s in range(500)]      # A/B：真的变差了

fp = sum(1 for r, _ in aa if r == 'reject_h0') / len(aa)
tp = sum(1 for r, _ in ab if r == 'reject_h0') / len(ab)
n_ab = sum(n for r, n in ab if r == 'reject_h0') / max(1, sum(1 for r, _ in ab if r == 'reject_h0'))
print(f'SPRT（允许任意时刻停止）:')
print(f'  A/A 假阳性率 {fp:.1%}  (名义 α=5%)')
print(f'  A/B 检出率   {tp:.1%}  (名义 1-β=80%)')
print(f'  检出坏版本平均只需 {n_ab:.0f} 个样本')
assert fp < 0.10, f'SPRT 应把假阳性控制在名义水平附近，得到 {fp:.1%}'
assert tp > 0.75, f'SPRT 应有足够检出力，得到 {tp:.1%}'
assert fp < fp_peek, 'SPRT 的假阳性应远低于朴素 peeking'
print(f'\n✅ SPRT 把假阳性从 {fp_peek:.0%} 压回 {fp:.0%}，同时保留了「随时可以停」的能力')

### 修复二：质量指标要用**非劣性**检验

发布要问的不是「新版本更好吗」，而是「**新版本会不会更差**」。
非劣性检验把原假设设成「差了至少 δ」，拒绝它才放行。

In [ ]:
def non_inferiority(wins_new, n, delta=0.05, z_crit=1.645):
    '''检验 H0: p <= 0.5 - delta（新版本明显更差） vs H1: p > 0.5 - delta。
       返回 (是否通过, z)。'''
    p = wins_new / n
    p0 = 0.5 - delta
    se = math.sqrt(p0 * (1 - p0) / n)
    z = (p - p0) / se
    return z > z_crit, z

for wins, n, label in [(500, 1000, '完全打平 (50%)'),
                       (470, 1000, '略差 (47%)'),
                       (430, 1000, '明显更差 (43%)'),
                       (48,  100,  '打平但样本太少')]:
    ok, z = non_inferiority(wins, n, delta=0.05)
    print(f'{label:<20s} 胜率 {wins/n:>5.1%}  z={z:>6.2f}  -> {"放行 ✅" if ok else "拦截 ❌"}')

assert non_inferiority(500, 1000)[0],  '打平应放行'
assert not non_inferiority(430, 1000)[0], '明显更差应拦截'
assert not non_inferiority(48, 100)[0], '样本太少时应拦截（不是「没发现问题」就放行）'
print('\n✅ 关键：样本不足时非劣性检验会**拦截**，而普通显著性检验会「没发现差异」而放行。')
print('   这个方向的翻转，是发布决策与科研检验最本质的区别。')

## 3 · HPA：控制律、纯延迟与震荡

$$R_{desired} = \lceil R_{current} \times M_{current} / M_{target} \rceil$$

带纯延迟的反馈系统会震荡。先把震荡跑出来，再用**稳定窗口**把它阻尼掉。

In [ ]:
def hpa_sim(traffic, mu=2.5, target_util=0.7, cold_start_ticks=12,
            count_pending=False, tolerance=0.0, stabilize_down_ticks=0,
            max_scale_down_frac=1.0, max_scale_up_frac=None, r0=8):
    '''闭环仿真。traffic: 每 tick 的 QPS 序列。返回 (副本轨迹, 过载 tick 数)。
       count_pending=False 复现最常见的失稳来源：**控制器无视正在启动的副本，持续重复下单**。'''
    ready, pending = r0, []            # pending: [就绪时刻]
    recent_desired, trace, overload = [], [], 0
    for t, lam in enumerate(traffic):
        arrived = [p for p in pending if p <= t]
        pending = [p for p in pending if p > t]
        ready += len(arrived)
        capacity = ready * mu
        util = lam / capacity if capacity else 10.0
        if util > 1.0:
            overload += 1
        ratio = util / target_util
        desired = ready if abs(ratio - 1.0) <= tolerance else math.ceil(ready * ratio)
        desired = max(1, desired)
        recent_desired.append(desired)
        # 已在路上的副本算不算数？这一行就是稳定与失稳的分水岭
        committed = ready + len(pending) if count_pending else ready
        if desired > committed:                                   # 扩容：立刻
            add = desired - committed
            if max_scale_up_frac is not None:
                add = min(add, max(1, math.ceil(ready * max_scale_up_frac)))
            for _ in range(add):
                pending.append(t + cold_start_ticks)
        elif desired < ready:                                     # 缩容：看稳定窗口
            safe = max(recent_desired[-(stabilize_down_ticks + 1):])
            if safe < ready:
                floor_ = math.ceil(ready * (1 - max_scale_down_frac))
                ready = max(1, max(safe, floor_))
        trace.append(ready + len(pending))
    return trace, overload

# 阶跃流量：20 QPS 突然涨到 100 QPS，再回落
traffic = [20]*20 + [100]*60 + [20]*40
naive,  ov_n = hpa_sim(traffic, count_pending=False, tolerance=0.0,
                       stabilize_down_ticks=0, max_scale_down_frac=1.0)
damped, ov_d = hpa_sim(traffic, count_pending=True, tolerance=0.10,
                       stabilize_down_ticks=20, max_scale_down_frac=0.10,
                       max_scale_up_frac=1.0)

def oscillation(tr):
    return sum(abs(tr[i] - tr[i-1]) for i in range(1, len(tr)))   # 总变动量

need = math.ceil(100 / (2.5 * 0.7))       # 峰值真正需要的副本数
print(f'峰值真实需求 = {need} 副本\n')
print(f'{"配置":<30s} {"峰值副本":>8s} {"总变动量":>9s} {"过载tick":>9s}')
print(f'{"无视 pending / 无容差 / 无窗口":<30s} {max(naive):>8d} {oscillation(naive):>9d} {ov_n:>9d}')
print(f'{"计入 pending+容差+窗口+限速":<30s} {max(damped):>8d} {oscillation(damped):>9d} {ov_d:>9d}')

assert max(naive) > 2 * need, f'朴素控制器应严重过冲（需要 {need}，下单 {max(naive)}）'
assert max(damped) < max(naive), '阻尼配置的过冲应明显更小'
assert oscillation(damped) < oscillation(naive), '阻尼配置应显著降低总变动量'
print(f'\n✅ 朴素控制器把 {need} 个副本的需求下成了 {max(naive)} 个 —— 因为它在冷启动的 3 分钟里'
      f'\n   一直看到「容量不足」，于是每个采样周期都重复下单。这就是经典的**积分饱和**。')
print(f'   计入 pending + 容差带 + 稳定窗口 + 限速后，过冲降到 {max(damped)}，'
      f'总变动量降低 {(1-oscillation(damped)/oscillation(naive)):.0%}')

### 冷启动是震荡的放大器

同样的控制器，冷启动越长越不稳定 —— 这是纯延迟反馈系统的普遍规律。

In [ ]:
print(f"{'冷启动(s)':>10s} {'峰值副本':>8s} {'总变动量':>9s} {'过载tick':>9s}")
prev_osc = 0
for cs_ticks in [1, 4, 12, 24]:      # tick=15s -> 15s / 60s / 180s / 360s
    tr, ov = hpa_sim(traffic, cold_start_ticks=cs_ticks, stabilize_down_ticks=0)
    print(f'{cs_ticks*15:>10d} {max(tr):>8d} {oscillation(tr):>9d} {ov:>9d}')
    if cs_ticks == 1: prev_osc = oscillation(tr)

tr_fast, ov_fast = hpa_sim(traffic, cold_start_ticks=1,  stabilize_down_ticks=0)
tr_slow, ov_slow = hpa_sim(traffic, cold_start_ticks=24, stabilize_down_ticks=0)
assert max(tr_slow) > max(tr_fast), '冷启动越长，过冲越严重'
assert ov_slow > ov_fast, '冷启动越长，过载时间越久'
print(f'\n✅ 冷启动 15s vs 360s：峰值副本 {max(tr_fast)} -> {max(tr_slow)}，'
      f'过载时长 {ov_fast} -> {ov_slow} tick')
print('   **模块 01 压缩的每一秒冷启动，都在这里变成更稳的扩缩和更少的超配。**')

## 4 · 热备容量：反应式扩容够不够？

$$R_{buffer} \ge \frac{1}{\mu}\cdot\frac{d\lambda}{dt}\cdot T_{cold}$$

这个公式回答一个很实际的问题：**要养几个空闲副本，才能在扩容生效前撑住**。

In [ ]:
def buffer_replicas(dlambda_dt, t_cold_s, mu):
    return math.ceil(dlambda_dt * t_cold_s / mu)

print(f"{'场景':<22s} {'dλ/dt':>8s} {'T_cold':>8s} {'需要热备':>9s} {'月成本$':>10s}")
GPU_PER_REPLICA, GPU_HOURLY, MU = 2, 4.0, 2.5
for name, rate, tc in [('平缓日周期', 0.05, 180), ('中等推广', 0.5, 180),
                       ('营销开闸', 2.0, 180), ('营销开闸(冷启动30s)', 2.0, 30)]:
    b = buffer_replicas(rate, tc, MU)
    cost = b * GPU_PER_REPLICA * GPU_HOURLY * 24 * 30
    print(f'{name:<22s} {rate:>8.2f} {tc:>8d} {b:>9d} {cost:>10,.0f}')

b180 = buffer_replicas(2.0, 180, MU)
b30  = buffer_replicas(2.0, 30,  MU)
assert b180 == 6 * b30, '冷启动缩短 6 倍，热备需求也降到 1/6'
assert b180 > 100, '陡峭尖峰下热备需求不现实 —— 说明必须换手段'
print(f'\n✅ 两条结论：')
print(f'   ① 陡峭尖峰（{b180} 个热备）靠热备买不起 -> 必须预约扩容 / 限流降级 / 上游整形。')
print(f'   ② 冷启动是热备成本的**乘数**：180s->30s 让热备从 {b180} 降到 {b30} 个。')

## ✏️ 练习 1：错误预算燃烧率与自动回滚

实现 `burn_rate(error_rate, slo)` 和 `rollback_decision(burn, windows_exceeded, cooldown_active)`。
- `burn_rate = error_rate / (1 - slo)`
- 回滚规则：`cooldown_active` 为真 → 永不自动回滚（返回 `'manual'`）；
  否则 `burn >= 100 且 windows_exceeded >= 2` → `'auto_rollback'`；
  `burn >= 14.4` → `'page'`；`burn >= 6` → `'alert'`；否则 `'ok'`。

In [ ]:
def burn_rate(error_rate, slo):
    # TODO
    raise NotImplementedError

def rollback_decision(burn, windows_exceeded, cooldown_active):
    # TODO
    raise NotImplementedError

In [ ]:
# —— 练习 1 自测 ——
assert abs(burn_rate(0.01, 0.999) - 10.0) < 1e-9, '1% 错误率 / 0.1% 预算 = 10 倍'
assert abs(burn_rate(0.01, 0.99) - 1.0) < 1e-9,   '同样 1%，对 99% SLO 只是 1 倍'
assert rollback_decision(150, 2, False) == 'auto_rollback'
assert rollback_decision(150, 1, False) == 'page', '只超一个窗口不自动回滚（防抖）'
assert rollback_decision(150, 5, True)  == 'manual', '冷却期内绝不自动回滚（防回滚风暴）'
assert rollback_decision(20, 3, False)  == 'page'
assert rollback_decision(8,  3, False)  == 'alert'
assert rollback_decision(1,  9, False)  == 'ok'
print('✅ 练习 1 通过：burn rate 把阈值归一化到 SLO，一套规则适配所有服务')

## ✏️ 练习 2：金丝雀阶段规划

实现 `canary_plan(stages, qps, min_samples)`：`stages` 是 `[(流量比例, 计划分钟数), ...]`。
对每个阶段计算实际样本量 `qps * 60 * minutes * frac`；
若不足 `min_samples`，**延长该阶段**到刚好够（分钟数向上取整）。
返回 `[(frac, actual_minutes, samples), ...]`。

In [ ]:
def canary_plan(stages, qps, min_samples):
    # TODO: 对每个 (frac, minutes)：
    #   need_min = ceil(min_samples / (qps*60*frac))；actual = max(minutes, need_min)
    #   samples = int(qps*60*actual*frac)
    raise NotImplementedError

In [ ]:
# —— 练习 2 自测 ——
plan = canary_plan([(0.01, 5), (0.05, 10), (0.25, 20)], qps=50, min_samples=1000)
for frac, mins, n in plan:
    print(f'  {frac:>5.0%} 流量, {mins:>3d} 分钟, {n:>7,d} 样本')
assert all(n >= 1000 for _, _, n in plan), '每个阶段都必须达到最小样本量'
assert plan[0][1] > 5, '1% 流量 × 50 QPS，5 分钟只有 150 个样本，必须延长'
assert plan[2][1] == 20, '25% 流量下 20 分钟已足够，不该延长'
total_min = sum(m for _, m, _ in plan)
print(f'总耗时 {total_min} 分钟')
assert total_min > 35, '样本量约束会让金丝雀比「计划表」更慢 —— 这是对的'
print('✅ 练习 2 通过：金丝雀的真实时长由**样本量**决定，不是由计划表决定')

## ✏️ 练习 3：HPA 的期望副本数

实现 `hpa_desired(current_replicas, current_metric, target_metric, tolerance=0.1,
min_r=1, max_r=100)`：按 K8s 原始算法返回期望副本数。
规则：`ratio = current/target`；若 `|ratio - 1| <= tolerance` 返回 `current_replicas`（死区）；
否则 `ceil(current_replicas * ratio)`，最后夹到 `[min_r, max_r]`。

In [ ]:
def hpa_desired(current_replicas, current_metric, target_metric, tolerance=0.1, min_r=1, max_r=100):
    # TODO
    raise NotImplementedError

In [ ]:
# —— 练习 3 自测 ——
assert hpa_desired(10, 0.70, 0.70) == 10, '正中目标：不动'
assert hpa_desired(10, 0.73, 0.70) == 10, '偏差 4% 在容差带内：不动'
assert hpa_desired(10, 0.90, 0.70) == 13, 'ceil(10 * 0.9/0.7) = 13'
assert hpa_desired(10, 0.35, 0.70) == 5,  '利用率减半 -> 副本减半'
assert hpa_desired(10, 0.01, 0.70, min_r=2) == 2, '受 minReplicas 约束'
assert hpa_desired(10, 9.99, 0.70, max_r=50) == 50, '受 maxReplicas 约束'
# 容差带的价值：在目标附近抖动时不产生动作
noisy = [hpa_desired(10, 0.70 + d, 0.70) for d in (-0.05, -0.02, 0.02, 0.05)]
assert all(r == 10 for r in noisy), '目标附近的噪声不应触发扩缩'
print('✅ 练习 3 通过：容差带（死区）是 HPA 稳定性的第一道保险')

---
### 📖 参考答案（先自己做，再对照）

In [ ]:
# 练习 1 参考答案
def burn_rate(error_rate, slo):
    return error_rate / (1 - slo)

def rollback_decision(burn, windows_exceeded, cooldown_active):
    if cooldown_active:                             return 'manual'
    if burn >= 100 and windows_exceeded >= 2:       return 'auto_rollback'
    if burn >= 14.4:                                return 'page'
    if burn >= 6:                                   return 'alert'
    return 'ok'

In [ ]:
# 练习 2 参考答案
def canary_plan(stages, qps, min_samples):
    out = []
    for frac, minutes in stages:
        need_min = math.ceil(min_samples / (qps * 60 * frac))
        actual = max(minutes, need_min)
        out.append((frac, actual, int(qps * 60 * actual * frac)))
    return out

In [ ]:
# 练习 3 参考答案
def hpa_desired(current_replicas, current_metric, target_metric, tolerance=0.1, min_r=1, max_r=100):
    ratio = current_metric / target_metric
    if abs(ratio - 1.0) <= tolerance:
        return current_replicas
    return max(min_r, min(max_r, math.ceil(current_replicas * ratio)))

---
## 🧪 真实数据胶囊：四种发布策略的总代价

场景：13 副本 × 2 卡 × $4/卡·h，新版本有 bug（错误率 5%），SLO=99.9%。
把「额外资源成本」与「错误预算消耗」放在一张表上比。

In [ ]:
REPLICAS, GPU_PER_REPLICA, GPU_HOURLY, SLO = 13, 2, 4.0, 0.999
BUDGET_MIN = (1 - SLO) * 30 * 24 * 60      # 月度错误预算（分钟）

def strategy_cost(extra_replicas, duration_min, affected_frac, impact_min):
    gpu_cost = extra_replicas * GPU_PER_REPLICA * GPU_HOURLY * (duration_min / 60)
    budget_min = affected_frac * impact_min
    return gpu_cost, budget_min, budget_min / BUDGET_MIN

strategies = [
    ('Recreate',            0,        15, 1.00, 15),
    ('RollingUpdate 25%',   3,        15, 0.50, 15),
    ('Blue-Green',          REPLICAS, 20, 1.00, 3),
    ('Canary 5% + 自动分析', 1,        25, 0.05, 8),
]
print(f'月度错误预算: {BUDGET_MIN:.1f} 分钟\n')
print(f"{'策略':<22s} {'额外$':>8s} {'预算消耗(min)':>14s} {'占月度预算':>11s}")
res = {}
for name, er, dm, af, im in strategies:
    g, b, pct = strategy_cost(er, dm, af, im)
    res[name] = (g, b)
    print(f'{name:<22s} {g:>8.2f} {b:>14.2f} {pct:>10.1%}')

canary_g, canary_b = res['Canary 5% + 自动分析']
recreate_b = res['Recreate'][1]
bg_g = res['Blue-Green'][0]
assert canary_b < recreate_b / 20, '金丝雀的预算消耗应比 Recreate 低一个数量级以上'
assert canary_g < bg_g / 5, '金丝雀的额外资源成本应远低于蓝绿'
print(f'\n✅ 金丝雀：预算消耗比 Recreate 低 {recreate_b/canary_b:.0f} 倍，'
      f'额外成本比蓝绿低 {bg_g/canary_g:.0f} 倍。')
print('   它唯一的代价是**工程复杂度**（按权重分流的网关 + 自动分析管线）——一次投入，长期受益。')

**🧪 胶囊练习**：实现 `autoscale_savings(peak_qps, trough_qps, mu, target_rho, hours_at_peak)`：
对比「按峰值固定容量」与「自动扩缩」的月成本。
自动扩缩假设：峰值时段用峰值副本、其余时段用波谷副本。
返回 `(fixed_monthly, autoscale_monthly, saving_frac)`。

In [ ]:
def autoscale_savings(peak_qps, trough_qps, mu, target_rho, hours_at_peak):
    # TODO: r_peak = ceil(peak/(mu*rho)); r_trough = ceil(trough/(mu*rho))
    #   fixed = r_peak * GPU_PER_REPLICA * GPU_HOURLY * 24 * 30
    #   auto  = (r_peak*hours_at_peak + r_trough*(24-hours_at_peak)) * GPU_PER_REPLICA * GPU_HOURLY * 30
    #   返回 (fixed, auto, 1 - auto/fixed)
    raise NotImplementedError

In [ ]:
# 自测
fixed, auto, saving = autoscale_savings(200, 20, mu=2.5, target_rho=0.7, hours_at_peak=8)
print(f'固定容量 ${fixed:>10,.0f}/月')
print(f'自动扩缩 ${auto:>10,.0f}/月')
print(f'节省 {saving:.1%}')
assert auto < fixed, '自动扩缩应更便宜'
assert 0.4 < saving < 0.8, f'10:1 的日夜比应省 40%~80%，得到 {saving:.1%}'
# 波峰波谷差距小时，收益也小
_, _, small = autoscale_savings(100, 80, 2.5, 0.7, 8)
assert small < saving / 3, '流量曲线越平，自动扩缩越不值得做'
print(f'\n对比：日夜比 10:1 省 {saving:.0%}；日夜比 1.25:1 只省 {small:.0%}')
print('✅ 胶囊练习通过：**先看流量曲线，再决定要不要上自动扩缩**')

In [ ]:
# 📖 胶囊参考答案
def autoscale_savings(peak_qps, trough_qps, mu, target_rho, hours_at_peak):
    r_peak   = math.ceil(peak_qps   / (mu * target_rho))
    r_trough = math.ceil(trough_qps / (mu * target_rho))
    unit = GPU_PER_REPLICA * GPU_HOURLY
    fixed = r_peak * unit * 24 * 30
    auto  = (r_peak * hours_at_peak + r_trough * (24 - hours_at_peak)) * unit * 30
    return fixed, auto, 1 - auto / fixed

---
## 🔧 旁注：真实系统里这些对应什么

- **滚动更新参数** → `spec.strategy.rollingUpdate.{maxSurge,maxUnavailable}`；LLM 服务标准配置是 `maxUnavailable: 0`。
- **金丝雀 + 自动分析** → Argo Rollouts 的 `Rollout` + `AnalysisTemplate`，或 Flagger 的 `Canary` CRD。判据写成 Prometheus 查询。
- **序贯检验** → 目前主流工具还没内置；实践中的近似是「多窗口 + 要求连续 N 次超阈值」。要严格做需自己实现 analysis provider。
- **非劣性检验** → 离线评测管线（C03/C10）的判据，在发布流水线里作为质量门禁。
- **HPA 控制律** → `autoscaling/v2` 的 `HorizontalPodAutoscaler`，`behavior.scaleDown.stabilizationWindowSeconds: 300` 是默认阻尼。
- **按队列深度扩缩** → KEDA 的 `ScaledObject`（支持 Prometheus/Kafka/SQS 等外部触发器），原生 HPA 需配 prometheus-adapter。
- **burn rate 告警** → Prometheus 多窗口多燃烧率规则；Sloth / Pyrra 能从 SLO 定义自动生成这些规则。

你在这里仿真出的震荡曲线，和真实集群 `kubectl get hpa -w` 看到的副本数抖动是同一个现象。

### 小结
- 发布的全部目的是压小 **爆炸半径 × MTTR**；先压爆炸半径（便宜、收益大），再压 MTTR。
- **maxUnavailable=0** 是 LLM 服务的标准配置（容量永不下探）；maxSurge 是「用少量 GPU 买更新速度」的极便宜交易。
- 金丝雀判据必须处理 **peeking**：朴素反复检验会把假阳性率抬到 20%+，SPRT 能压回名义水平；质量指标要用**非劣性**检验（样本不足时应拦截而非放行）。
- HPA 是带纯延迟的比例控制器，**冷启动是震荡的放大器**；用稳定窗口+容差带+缩容限速阻尼，用**队列深度**（领先指标）而非 GPU 利用率或延迟（滞后指标）触发。
- **热备容量 = dλ/dt × T_cold / μ**：陡峭尖峰买不起热备，必须预约扩容/限流降级；而冷启动是这笔账的乘数。
- 自动回滚由 **burn rate** 驱动，并且必须有冷却期与次数上限来防回滚风暴。

下一站：**模块 05 · 云平台、集群调度与成本** —— 这一切跑在谁家的机器上，一个月到底多少钱？